In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

#### generate text

> `gpt-4.1-mini` is fast & cost effective

> `temperature` controls randomness

> ranges from 0 to 1 (0: low randomness, 1: high randomness)

> if no. of `max_tokens` are not given, model decides when to stop

In [ ]:
client = OpenAI(api_key = OPENAI_API_KEY)

In [ ]:
def generate_text(prompt, max_tokens, temperature):
    response = client.responses.create(
        model = "gpt-4.1-mini",
        input = prompt,
        max_output_tokens = max_tokens,
        temperature = temperature
    )

    return response.output_text

In [ ]:
prompt = "Once upon a time in a village"
generate_text(prompt, 100, 0.8)

In [ ]:
prompt = "What is a black hole?"
generate_text(prompt, 100, 0.4)

#### summarize text

In [ ]:
def text_summarization(prompt, max_tokens, temperature):
    response = client.responses.create(
        model = "gpt-4.1-mini",

        input = [
            # instruction
            {
                "role": "system",
                "content": "You will be provided a paragraph and your task is to summarize it into bullet points."
            },

            # example 1
            {
                "role": "user",
                "content": "Cloud computing allows users to store and access data over the internet instead of local storage. It offers scalability, cost-efficiency, and remote accessibility."
            },

            {
                "role": "assistant",
                "content": "- Stores data over the internet\n- Provides scalability\n- Cost-efficient solution\n- Enables remote access"
            },

            # example 2
            {
                "role": "user",
                "content": "Cybersecurity involves protecting systems, networks, and data from digital attacks. It includes practices like encryption, authentication, and monitoring to ensure data safety."
            },

            {
                "role": "assistant",
                "content": "- Protects systems and data\n- Prevents digital attacks\n- Uses encryption and authentication\n- Ensures data safety"
            },

            # input
            {
                "role": "user",
                "content": prompt
            }
        ],

        max_output_tokens = max_tokens,
        temperature = temperature
    )

    return response.output_text


In [ ]:
prompt = "Climate change refers to long-term shifts in temperature and weather patterns, primarily caused by human activities such as burning fossil fuels and deforestation. These activities increase greenhouse gas concentrations, leading to global warming. The effects include rising sea levels, extreme weather events, and loss of biodiversity. Addressing climate change requires global cooperation, adoption of renewable energy, and sustainable practices."

text_summarization(prompt, 100, 0.2) # low temperature since we need less creativity and the output needs to stick to the example format

#### poetic chatbot

In [ ]:
def poetic_chatbot(prompt, max_tokens, temperature):
    response = client.responses.create(
        model="gpt-4.1-mini",

        input = [
            # instruction
            {
                "role": "system",
                "content": "You are a poetic chatbot."
            },

            # example 1
            {
                "role": "user",
                "content": "When was Google founded?"
            },

            {
                "role": "assistant",
                "content": "In the late '90s, a spark did ignite, Google emerged, a radiant light. By Larry and Sergey, in '98, it was born, a search engine new, on the web it was sworn."
            },

            # example 2
            {
                "role": "user",
                "content": "Which country has the youngest president?"
            },
            {
                "role": "assistant",
                "content": "Ah, the pursuit of youth in politics, a theme we explore. In Austria, Sebastian Kurz did implore, at the age of 31, his journey did begin, leading with vigor, in a world filled with din."
            },

            # input
            {
                "role": "user",
                "content": prompt
            }
        ],

        max_output_tokens = max_tokens,
        temperature = temperature,
        
    )
    
    return response.output_text

In [ ]:
prompt = "What is a black hole?"
poetic_chatbot(prompt, 100, 1) # high temperature as we need more creativity for a poetic output

#### LangChain

> it helps build applications that use LLMs

> it connects LLMs with data and tools

##### load documents

> other document loaders can read `PDFs`, `GitHub Repos`, etc.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

In [ ]:
urls = [
    "https://en.wikipedia.org/wiki/Extinction",
    "https://en.wikipedia.org/wiki/Lists_of_extinct_species",
    "https://education.nationalgeographic.org/resource/resource-library-extinction/",
    "https://www.worldwildlife.org/resources/explainers/what-is-the-sixth-mass-extinction-and-what-can-we-do-about-it/"
]

In [ ]:
loader = WebBaseLoader(urls)

docs = loader.load()

In [ ]:
print(len(docs)) # files in repo

print(docs[0].metadata) # info about file 'x' in the reop

##### split documents

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

> `chunk size`: set max chars in a chunk, (LLMs work better with small chunks)

> `chunk overlap`: last chars of a chunk repeat in the next, (to avoid context breaks at the boundaries)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 100)

chunks = text_splitter.split_documents(docs)

In [ ]:
print(len(chunks))

print(chunks[0].metadata)
print(chunks[1].metadata)

##### store embeddings

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
embeddings = OpenAIEmbeddings(api_key = OPENAI_API_KEY)

vectorstore = FAISS.from_documents(chunks, embeddings)

##### conversation chain

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
llm = ChatOpenAI(
    api_key = OPENAI_API_KEY,
    model = "gpt-4.1-mini",
    temperature = 0.2
)

In [ ]:
chat_history = []

system_msg = {"role": "system", "content": "Answer the question using the context given. If the answer is not in the context, say 'I don't know'."}

> `prev_queries`:  get last 'k' user queries for follow up ques

> `RAG` needs prev queries too as it might not understand the context of follow up ques

> `k` in similarity search get top k relevant chunks

In [ ]:
def gpt_rag(query):
    prev_queries = " ".join([msg['content'] for msg in chat_history[-4:] if msg['role'] == 'user'])

    relevant_chunks = vectorstore.similarity_search(prev_queries + " " + query, k = 3)

    context = "\n\n".join(chunk.page_content for chunk in relevant_chunks)

    prompt = [system_msg] + chat_history + [{"role": "user", "content": f"Context: \n{context} \n\nQuestion: {query}"}]

    response = llm.invoke(prompt)

    chat_history.extend([{"role": "user", "content": query},
                        {"role": "assistant", "content": str(response.content)}])

    return response.content

In [ ]:
query = "What is mass extinction? How many species have been affected by it so far?"
gpt_rag(query)